> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Synthetic — analytical SE versus weighted bootstrap

In [ ]:
import os, numpy as np, pandas as pd, vigor, time
BOOTSTRAPS=int(os.environ.get('VIGOR_TUTORIAL_BOOTSTRAPS', os.environ.get('DOMINO_TUTORIAL_BOOTSTRAPS', 50)))
cfg=vigor.simulation_scenario("complex");cfg.update(dict(heritability=.55,compute_grms=True))
sim=vigor.simulate_genomic_dataset(n_samples=260,n_variants=800,n_traits=6,seed=42,trait_kwargs=cfg)
K=sim.grms.sel(kernel="additive").to_pandas().to_numpy(float);Y=sim.phenotypes.to_pandas()
t=time.perf_counter();a=vigor.genetic_varcov_PSD(K,Y,solver="lbfgs",rank_G="full",rank_E="full",variance_closure="auto",analytic_se="final",analytic_se_probes=16,analytic_trace_probes=8,n_boot=0,random_state=42,progress=False);ta=time.perf_counter()-t
t=time.perf_counter();b=vigor.genetic_varcov_PSD(K,Y,solver="lbfgs",rank_G="full",rank_E="full",variance_closure="auto",analytic_se=False,n_boot=BOOTSTRAPS,bootstrap_n_jobs=2,bootstrap_blas_threads=1,random_state=42,progress=False);tb=time.perf_counter()-t
print("analytic s",ta,f"bootstrap{BOOTSTRAPS} s",tb)
print("G SE rel-Fro",np.linalg.norm(np.asarray(a["G_se"].to_pandas())-np.asarray(b["G_se"].to_pandas()))/np.linalg.norm(np.asarray(a["G_se"].to_pandas())))


Publication regression benchmarking uses 600 bootstrap replicates; the tutorial uses 50 so it remains interactive.